# Simpson's Paradox: Electricity Access vs CO₂ Emissions
**Data source:** Gapminder via Our World in Data (OWID) / Global Carbon Project

This notebook:
1. Downloads electricity access and CO₂ per-capita data directly from OWID GitHub
2. Merges with income-group metadata for subgroup analysis
3. Demonstrates **Simpson's Paradox** — the aggregate trend reverses within income subgroups
4. Highlights the **USA in 2022** and projects its position in **2028** after the 2025 federal CO₂ policy reversal (Paris withdrawal + IRA rollback)

---
### Requirements
```bash
pip install pandas numpy matplotlib seaborn scipy requests
```

## 0 · Imports & Plot Style

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import requests
import io
import warnings
warnings.filterwarnings("ignore")

# ── Dark theme ──
plt.rcParams.update({
    "figure.facecolor": "#0f0f1a",
    "axes.facecolor":   "#1a1a2e",
    "axes.edgecolor":   "#444466",
    "axes.labelcolor":  "#ccccee",
    "xtick.color":      "#aaaacc",
    "ytick.color":      "#aaaacc",
    "text.color":       "#e0e0f0",
    "grid.color":       "#2a2a44",
    "grid.alpha":       0.5,
    "font.family":      "DejaVu Sans",
    "font.size":        11,
})

INCOME_COLORS = {
    "Low income":          "#4e7af0",
    "Lower middle income": "#7ea1d1",
    "Upper middle income": "#a7cb7d",
    "High income":         "#f0a500",
}

print("✅ Imports complete")

## 1 · Download Data
We pull two CSV files directly from OWID's public GitHub repos — no API key needed.

| Dataset | Source |
|---------|--------|
| CO₂ per capita, GDP, population, renewable electricity share | [owid/co2-data](https://github.com/owid/co2-data) |
| Access to electricity (%) | [owid/energy-data](https://github.com/owid/energy-data) |

In [ ]:
print("⬇  Downloading CO₂ data …")
CO2_URL = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"
r = requests.get(CO2_URL, timeout=30)
owid = pd.read_csv(io.StringIO(r.text), low_memory=False)

# Keep only country-level rows (OWID prefixes aggregates with "OWID")
owid = owid[owid["iso_code"].notna() & ~owid["iso_code"].str.startswith("OWID")]

co2_df = owid[["country", "iso_code", "year",
               "co2_per_capita",
               "electricity_demand",       # TWh
               "share_elec_renewables",    # %
               "gdp",                      # constant 2011 USD
               "population"]].copy()
co2_df = co2_df[co2_df["year"].between(2000, 2022)].dropna(subset=["co2_per_capita"])

print(f"   CO₂ dataset: {len(co2_df):,} country-year rows")

print("⬇  Downloading electricity access data …")
ENERGY_URL = "https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv"
r2 = requests.get(ENERGY_URL, timeout=30)
energy = pd.read_csv(io.StringIO(r2.text), low_memory=False)
energy = energy[energy["iso_code"].notna() & ~energy["iso_code"].str.startswith("OWID")]

elec_access = energy[["country", "iso_code", "year",
                       "access_to_electricity"]].dropna(subset=["access_to_electricity"])
elec_access = elec_access[elec_access["year"].between(2000, 2022)]

print(f"   Electricity access dataset: {len(elec_access):,} country-year rows")
print("✅ Downloads complete")

## 2 · Income Group Classification
We assign World Bank 2022 income groups using ISO-3 country codes.
This avoids a fragile third download and keeps the notebook self-contained.

In [ ]:
HIGH_INCOME = {
    "AUS","AUT","BEL","CAN","CHE","CHL","CZE","DEU","DNK","ESP","EST","FIN",
    "FRA","GBR","GRC","HUN","IRL","ISL","ISR","ITA","JPN","KOR","LTU","LUX",
    "LVA","NLD","NOR","NZL","POL","PRT","SAU","SVK","SVN","SWE","USA","ARE",
    "BHR","KWT","QAT","SGP","HKG","TWN","MKD","HRV","CYP","MLT","ROU","BGR",
    "OMN","TTO","URY","PAN","MYS","BRN","ANT","NCL","PYF","GUM","VIR","BMU",
    "CYM","GIB","MAC","BHS","BRB","ATG","LCA","VCT","GRD","KNA","DMA","TCA",
    "ABW","CUW","SXM","AIA","MSR","TUV","PLW","MHL","NRU","FSM","COK","NIU",
    "AND","LIE","MCO","SMR","VAT"
}
UPPER_MIDDLE = {
    "ARG","AZE","BLR","BIH","BOL","BRA","CHN","COL","CRI","CUB","DOM","DZA",
    "ECU","EGY","FJI","GAB","GEO","GNQ","GTM","GUY","IDN","IRN","IRQ","JAM",
    "JOR","KAZ","LBN","LBY","LKA","MDA","MDV","MEX","MNE","MNG","MUS","NAM",
    "NOR","PRY","PER","RUS","SRB","SUR","THA","TKM","TON","TUN","TUR","UKR",
    "VEN","ZAF","ARM","ALB","BWA","CPV","CMR","COG","CIV","GHA","HND","MKD",
    "NIC","PNG","WSM","SLV","SWZ","UZB","VNM","XKX","BLZ","AGO","KGZ","TJK",
    "MMR","KIR","VUT","PSE","WLF"
}
LOW_MIDDLE = {
    "AFG","BEN","BGD","BFA","BTN","CIV","CMR","COD","COM","DJI","ERI","ETH",
    "GHA","GIN","GMB","GNB","HTI","HND","IND","KEN","KHM","KIR","LAO","LBR",
    "LSO","MAR","MDG","MLI","MMR","MOZ","MRT","MWI","NER","NGA","NIC","NPL",
    "PAK","PHL","PNG","SEN","SLB","SLE","SLV","SDN","SOM","SSD","STP","SWZ",
    "SYR","TCD","TGO","TJK","TLS","TZA","UGA","UKR","VNM","VUT","WSM","YEM",
    "ZMB","ZWE","CAF","COD","GNB","SOM","SSD","ERI","SLE","LBR","MDG","MWI",
    "MOZ","BEN","NER","MLI","TCD","BFA","RWA","TZA","UGA","ZMB","ZWE","HTI",
    "KGZ","KHM","LAO","MYA","NPL","PHI","TJK","UZB"
}

def assign_income(iso):
    if iso in HIGH_INCOME:   return "High income"
    if iso in UPPER_MIDDLE:  return "Upper middle income"
    if iso in LOW_MIDDLE:    return "Lower middle income"
    return "Low income"

print("✅ Income group classifier defined")

## 3 · Merge & Clean

In [ ]:
merged = pd.merge(
    co2_df, elec_access,
    on=["country", "iso_code", "year"], how="inner"
).dropna(subset=["co2_per_capita", "access_to_electricity"])

merged["income_group"]    = merged["iso_code"].apply(assign_income)
merged["gdp_per_capita"]  = merged["gdp"] / merged["population"]

# Focus on a single representative year (pre-COVID, rich data coverage)
FOCUS_YEAR = 2019
df = merged[merged["year"] == FOCUS_YEAR].copy()

print(f"✅ {len(df)} countries with full data for {FOCUS_YEAR}")
print()
print("Income group distribution:")
print(df["income_group"].value_counts().to_string())

## 4 · US Position: 2022 Actual & 2028 Projections

**2022 actual:** US CO₂ per capita ≈ 14.21 t (EIA / OWID), electricity access = 100%

**2025 policy shift context:**
- January 2025: US withdraws from Paris Agreement (second time)
- July 2025: "One Big Beautiful Bill" rolls back IRA clean energy tax credits
- EPA emissions standards relaxed for power sector and transportation

**2028 scenarios** (illustrative, based on Climate Action Tracker 2025 analysis):
- **High reversal** (~15.0 t): Full policy effect, fossil subsidies reinstated, EV adoption stalls
- **Low reversal** (~14.5 t): Market forces (cheap solar, EVs) partially offset policy rollback

In [ ]:
US_2022      = {"co2": 14.21, "elec": 100.0, "label": "USA 2022 (actual)"}
US_2028_HIGH = {"co2": 15.0,  "elec": 100.0, "label": "USA 2028 — policy reversal (full effect est.)"}
US_2028_LOW  = {"co2": 14.5,  "elec": 100.0, "label": "USA 2028 — policy reversal (market offset est.)"}

print("US scenarios defined:")
for s in [US_2022, US_2028_HIGH, US_2028_LOW]:
    print(f"  {s['label']}: {s['co2']} t CO₂/cap")

## 5 · Regression Helper

In [ ]:
from scipy import stats

def ols(x, y):
    """OLS regression; returns scipy LinregressResult or None if < 5 finite points."""
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 5:
        return None
    return stats.linregress(x[mask], y[mask])

x_col = "access_to_electricity"
y_col  = "co2_per_capita"
order  = ["Low income", "Lower middle income", "Upper middle income", "High income"]

# Pre-compute aggregate regression used across multiple figures
reg_all = ols(df[x_col].values, df[y_col].values)
print(f"Aggregate slope: {reg_all.slope:+.4f}  (R²={reg_all.rvalue**2:.3f}, p={reg_all.pvalue:.2e})")

## 6 · Figure 1 — Simpson's Paradox: Main Scatter Plot
The **white line** is the aggregate OLS fit across all countries.  
The **dashed coloured lines** are within-group fits.  
Notice how the aggregate slope is positive (more electricity → more CO₂) but this reverses or flattens within income groups — especially for high-income countries.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))

for grp in order:
    sub = df[df["income_group"] == grp]
    ax.scatter(sub[x_col], sub[y_col],
               color=INCOME_COLORS[grp], alpha=0.65, s=60,
               label=grp, zorder=3, edgecolors="none")
    reg = ols(sub[x_col].values, sub[y_col].values)
    if reg:
        xs = np.linspace(sub[x_col].min(), sub[x_col].max(), 100)
        ax.plot(xs, reg.slope * xs + reg.intercept,
                color=INCOME_COLORS[grp], linewidth=2, linestyle="--", alpha=0.9, zorder=4)

# Aggregate line
xs_all = np.linspace(df[x_col].min(), df[x_col].max(), 200)
ax.plot(xs_all, reg_all.slope * xs_all + reg_all.intercept,
        color="white", linewidth=2.5, linestyle="-", alpha=0.9,
        label=f"Aggregate (slope={reg_all.slope:.3f})", zorder=5)

# USA 2022
ax.scatter(US_2022["elec"], US_2022["co2"], color="#ff4444", s=200, zorder=10,
           marker="*", edgecolors="white", linewidths=1)
ax.annotate(US_2022["label"], xy=(US_2022["elec"], US_2022["co2"]),
            xytext=(-110, 12), textcoords="offset points", color="#ff6666",
            fontsize=9, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="#ff6666", lw=1.2))

# USA 2028 HIGH
ax.scatter(US_2028_HIGH["elec"], US_2028_HIGH["co2"], color="#ff9900", s=200,
           zorder=10, marker="^", edgecolors="white", linewidths=1)
ax.annotate(US_2028_HIGH["label"], xy=(US_2028_HIGH["elec"], US_2028_HIGH["co2"]),
            xytext=(-150, 22), textcoords="offset points", color="#ffaa33", fontsize=8.5,
            arrowprops=dict(arrowstyle="->", color="#ffaa33", lw=1.2))

# USA 2028 LOW
ax.scatter(US_2028_LOW["elec"], US_2028_LOW["co2"], color="#ffcc44", s=160,
           zorder=10, marker="^", edgecolors="white", linewidths=1)
ax.annotate(US_2028_LOW["label"], xy=(US_2028_LOW["elec"], US_2028_LOW["co2"]),
            xytext=(-160, -28), textcoords="offset points", color="#ffdd66", fontsize=8.5,
            arrowprops=dict(arrowstyle="->", color="#ffdd66", lw=1.2))

# Arrow 2022 → 2028
ax.annotate("", xy=(US_2028_HIGH["elec"], US_2028_HIGH["co2"]),
            xytext=(US_2022["elec"], US_2022["co2"]),
            arrowprops=dict(arrowstyle="-|>", color="white", lw=1.5, linestyle="dotted"))

textstr = (
    "Simpson's Paradox:\n"
    f"► Aggregate slope = {reg_all.slope:.3f}  →  more access = more CO₂\n\n"
    "► Within each income group (dashed):\n"
    "   slope ≈ 0 or negative for high-income.\n"
    "   The aggregate slope is driven by\n"
    "   group composition, not causality."
)
props = dict(boxstyle="round,pad=0.6", facecolor="#0a0a1a", alpha=0.85, edgecolor="#555577")
ax.text(0.02, 0.97, textstr, transform=ax.transAxes, fontsize=8.8,
        verticalalignment="top", bbox=props, color="#ccccee")

ax.set_xlabel("Access to Electricity  (%)", fontsize=13)
ax.set_ylabel("CO₂ Emissions per Capita  (tonnes)", fontsize=13)
ax.set_title(
    "Simpson's Paradox — Electricity Access vs CO₂ Emissions\n"
    f"Gapminder / OWID data · {FOCUS_YEAR} · n={len(df)} countries",
    fontsize=14, pad=14)
ax.legend(loc="upper left", bbox_to_anchor=(0.02, 0.62), framealpha=0.3, fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-2, 105)
ax.set_ylim(-0.5, 32)
plt.tight_layout()
plt.savefig("fig1_simpsons_paradox_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## 7 · Figure 2 — Slope Comparison by Income Group
A bar chart making the paradox **quantitatively explicit**: the aggregate slope is positive, but within-group slopes are near zero or negative for high-income countries.

In [ ]:
slope_data = {"Group": [], "Slope": [], "Color": []}
slope_data["Group"].append("All countries\n(aggregate)")
slope_data["Slope"].append(reg_all.slope)
slope_data["Color"].append("white")

for grp in order:
    sub = df[df["income_group"] == grp]
    reg = ols(sub[x_col].values, sub[y_col].values)
    if reg:
        slope_data["Group"].append(grp)
        slope_data["Slope"].append(reg.slope)
        slope_data["Color"].append(INCOME_COLORS[grp])

sdf = pd.DataFrame(slope_data)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(sdf["Group"], sdf["Slope"], color=sdf["Color"],
               edgecolor="#333355", linewidth=0.8, height=0.55)
ax.axvline(0, color="white", linewidth=1.2, alpha=0.6, linestyle="--")

for bar, val in zip(bars, sdf["Slope"]):
    ax.text(val + (0.003 if val >= 0 else -0.003),
            bar.get_y() + bar.get_height() / 2,
            f"{val:+.4f}", va="center",
            ha="left" if val >= 0 else "right",
            color="white", fontsize=9.5)

textstr2 = (
    "A positive aggregate slope suggests\n"
    "more electricity → more CO₂.\n\n"
    "But within high-income countries,\n"
    "the slope is near zero or negative —\n"
    "the paradox is the group-composition\n"
    "effect of mixing income tiers."
)
props2 = dict(boxstyle="round,pad=0.5", facecolor="#0a0a1a", alpha=0.85, edgecolor="#555577")
ax.text(0.98, 0.05, textstr2, transform=ax.transAxes, fontsize=8.5,
        va="bottom", ha="right", bbox=props2, color="#ccccee")

ax.set_xlabel("OLS Slope  (CO₂ per capita / % electricity access)", fontsize=11)
ax.set_title("Simpson's Paradox — Regression Slopes by Income Group\n"
             "Positive aggregate slope reverses or disappears in sub-groups",
             fontsize=12, pad=10)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("fig2_slopes_by_group.png", dpi=150, bbox_inches="tight")
plt.show()

## 8 · Figure 3 — US CO₂ Trajectory 2000–2028
Historical data from OWID extended with three forward scenarios:
- **Green dashed** — counterfactual Biden/IRA declining trajectory
- **Red dash-dot** — full policy reversal effect
- **Orange dotted** — market forces (cheap solar, EVs) partially offset the reversal

Key political events are annotated with vertical reference lines.

In [ ]:
us_ts = merged[(merged["iso_code"] == "USA") & (merged["year"] >= 2000)].sort_values("year")
proj_years = [2023, 2024, 2025, 2026, 2027, 2028]

# Biden trajectory: linear extrapolation of 2015-2022 decline
us_2015_2022 = us_ts[us_ts["year"].between(2015, 2022)]
reg_us = ols(us_2015_2022["year"].values, us_2015_2022["co2_per_capita"].values)
biden_proj = {y: reg_us.slope * y + reg_us.intercept for y in proj_years}

# Policy reversal scenarios
base = us_ts[us_ts["year"] == 2022]["co2_per_capita"].values[0]
reversal_high = {y: (base + 0.05*(y-2022) if y <= 2024
                     else base + 0.10 + 0.15*(y-2024))
                 for y in proj_years}
reversal_low  = {y: base + 0.02*(y-2022) for y in proj_years}

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(us_ts["year"], us_ts["co2_per_capita"],
        color="#4488ff", linewidth=2.5, label="USA historical (OWID)")
ax.fill_between(us_ts["year"], us_ts["co2_per_capita"], alpha=0.15, color="#4488ff")
ax.plot(list(biden_proj.keys()), list(biden_proj.values()),
        color="#44cc88", linewidth=2, linestyle="--", label="Counterfactual: Biden/IRA trajectory")
ax.plot(list(reversal_high.keys()), list(reversal_high.values()),
        color="#ff4444", linewidth=2, linestyle="-.", label="Policy reversal — full effect (est.)")
ax.plot(list(reversal_low.keys()), list(reversal_low.values()),
        color="#ff9900", linewidth=2, linestyle=":", label="Policy reversal — market offset (est.)")

ax.scatter([2022], [base], color="#ff4444", s=180, zorder=10,
           marker="*", edgecolors="white", linewidths=1, label=f"USA 2022: {base:.2f} t")
ax.scatter([2028], [reversal_high[2028]], color="#ff4444", s=120, zorder=10,
           marker="^", edgecolors="white", linewidths=1)
ax.scatter([2028], [reversal_low[2028]],  color="#ff9900", s=120, zorder=10,
           marker="^", edgecolors="white", linewidths=1)

events = {
    2015: ("Paris Agr.\nsigned",         "#aaffaa"),
    2017: ("Trump\nwithdrawal 1",        "#ff8888"),
    2021: ("Biden\nrejoins",             "#88aaff"),
    2025: ("Trump\nwithdrawal 2\n+ IRA rollback", "#ff4444"),
}
for yr, (lbl, col) in events.items():
    ax.axvline(yr, color=col, linewidth=1.2, linestyle=":", alpha=0.7)
    ax.text(yr + 0.1, 22.5, lbl, color=col, fontsize=7.5, va="top")

note = ("Sources: OWID/Global Carbon Project (historical) · Climate Action Tracker 2025\n"
        "2028 projections are illustrative estimates based on policy scenario analysis.")
ax.text(0.01, 0.01, note, transform=ax.transAxes, fontsize=7.5, color="#aaaacc", va="bottom")

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("CO₂ per Capita  (tonnes)", fontsize=12)
ax.set_title("United States CO₂ Emissions per Capita — Historical & 2028 Projection\n"
             "After the 2025 Paris Agreement withdrawal and IRA rollback",
             fontsize=13, pad=12)
ax.legend(loc="upper right", fontsize=9, framealpha=0.3)
ax.set_xlim(2000, 2029)
ax.set_ylim(10, 24)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fig3_us_trajectory.png", dpi=150, bbox_inches="tight")
plt.show()

## 9 · Figure 4 — Faceted View by Income Group
One panel per income group. The **solid coloured line** is the within-group slope; the **dashed white line** is the aggregate slope for reference.  
The ★ marks the USA (visible only in the High Income panel).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for i, grp in enumerate(order):
    ax = axes[i]
    sub = df[df["income_group"] == grp]
    color = INCOME_COLORS[grp]

    ax.scatter(sub[x_col], sub[y_col], color=color, alpha=0.7, s=55,
               edgecolors="none", zorder=3)

    reg = ols(sub[x_col].values, sub[y_col].values)
    if reg:
        xs = np.linspace(sub[x_col].min(), sub[x_col].max(), 100)
        ax.plot(xs, reg.slope * xs + reg.intercept, color=color, linewidth=2.5,
                zorder=4, label=f"Within-group slope: {reg.slope:+.4f}")

    xs_a = np.linspace(sub[x_col].min(), sub[x_col].max(), 100)
    ax.plot(xs_a, reg_all.slope * xs_a + reg_all.intercept,
            color="white", linewidth=1.5, linestyle="--", alpha=0.5,
            label=f"Aggregate slope: {reg_all.slope:+.4f}")

    us_row = df[(df["iso_code"] == "USA") & (df["income_group"] == grp)]
    if not us_row.empty:
        ax.scatter(us_row[x_col], us_row[y_col], color="#ff4444", s=200,
                   marker="*", zorder=10, edgecolors="white", linewidths=1)
        ax.text(us_row[x_col].values[0] - 3, us_row[y_col].values[0] + 0.4,
                "USA", color="#ff6666", fontsize=8.5, fontweight="bold")

    ax.set_title(grp, fontsize=11, color=color, pad=6)
    ax.set_xlabel("Electricity Access (%)", fontsize=9)
    ax.set_ylabel("CO₂ / capita (t)", fontsize=9)
    ax.legend(fontsize=7.5, framealpha=0.25, loc="upper left")
    ax.grid(True, alpha=0.25)

fig.suptitle(f"Simpson's Paradox — Faceted by Income Group  ({FOCUS_YEAR})\n"
             "Dashed white = aggregate slope · Solid = within-group slope",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("fig4_faceted_by_income.png", dpi=150, bbox_inches="tight")
plt.show()

## 10 · Summary Statistics Table

In [ ]:
print("=" * 62)
print("SUMMARY: Slope Reversal Check (Simpson's Paradox)")
print("=" * 62)
print(f"{'Group':<25} {'N':>4}  {'Slope':>8}  {'R²':>6}  {'p-value':>10}")
print("-" * 62)

grp_all_reg = ols(df[x_col].values, df[y_col].values)
print(f"{'Aggregate':<25} {len(df):>4}  {grp_all_reg.slope:>8.4f}"
      f"  {grp_all_reg.rvalue**2:>6.3f}  {grp_all_reg.pvalue:>10.2e}")

for grp in order:
    sub = df[df["income_group"] == grp]
    reg = ols(sub[x_col].values, sub[y_col].values)
    if reg:
        print(f"{grp:<25} {len(sub):>4}  {reg.slope:>8.4f}"
              f"  {reg.rvalue**2:>6.3f}  {reg.pvalue:>10.2e}")

print("=" * 62)
print()
print("US snapshot:")
print(f"  2022 actual CO₂/cap : {US_2022['co2']:.2f} t  (electricity access: 100%)")
print(f"  2028 projection high: {US_2028_HIGH['co2']:.2f} t  (policy reversal, full effect)")
print(f"  2028 projection low : {US_2028_LOW['co2']:.2f} t  (market forces partially offset)")
print()
print("Data sources:")
print("  https://github.com/owid/co2-data")
print("  https://github.com/owid/energy-data")
print("  https://climateactiontracker.org/countries/usa/")
print()
print("✅ All figures saved to current directory.")